# Tutorial to run a full simulation

In this tutorial we will simulate the entire population of neutron stars all together, which means initializing the entire population formed by $N$ neutron stars from some initial conditions, evolve it in time and finally apply the survey models to select the detected neutron stars.
For this approach we will run the script `pypopsyn/simulator/simulate_population_full.py`.

The default input parameters of the simulation are specified in the `pypopsyn/simulator/config_simulator.py` file.
To simulate populations with different initial parameters, the user can directly modify the simulator configuration in `pypopsyn/simulator/config_simulator.py` or alternatively for a more programmatic way a JSON dictionary containing configuration overrides for the simulation parameters can be provided as a command line argument to this script:
```
python pypopsyn/simulator/simulate_population_full.py --save_dir output/sim_full --parameter_override parameter_override.json
```
For example you can set the number of neutron stars to simulate, the kick-velocity model, the parameters of the initial distribution of spin periods and magnetic fields and several other parameters.
This will generate a new directory `output/sim_full` if it does not exist, in which the simulation results will be saved.
The output consists of the following files:
* `initial_population.pkl.gz`containing the initial conditions.
* `final_population.pkl.gz` containing the final population properties.
* `.pkl.gz` files for each one of the modelled surveys containing the population detected by that survey.
* `.json` and `.log` files containing the timing profiles for the simulation, if enabled.
* `configuration.json` file containing the configuration parameters for reproducibility.

In [ ]:
import argparse
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pathlib
import sys
from matplotlib.collections import PathCollection
from matplotlib.legend_handler import HandlerPathCollection, HandlerLine2D

def update(handle, orig):
    handle.update_from(orig)
    handle.set_alpha(1)
    handle.set_markersize(3)

import utilities.plot_settings
from pypopsyn.simulator.config_simulator import cfg
import pypopsyn.simulator.stellar_dynamics.coordinate_conversions as cc
from pypopsyn.simulator.simulate_population_full import simulate_population

## Setup and run the simulation

Change some parameters in the imported configuration file.
For example here we can change the following:
1. `NS_number`, number of neutron stars to simulate.
2. `t_age_max`, the maximum age for the simulated neutron stars in [yr].
3. `B_initial_log10_mean` and `B_initial_log10_sigma`, the mean and standard deviation of the Gaussian distribution for the $\log_{10}$ of the initial magnetic field.
4. `P_initial_log10_mean` and `P_initial_log10_sigma`, the mean and standard deviation of the Gaussian distribution for the $\log_{10}$ of the initial spin period.
5. `a_late`, the power-law index for the late time decay of the magnetic field after $\sim 10^6$ yr.

NOTE: to have a realistic birth rate ($\sim 1$ neutron star per century) and observed population of neutron stars you should set the `NS_number` and the `t_age_max` accordingly. However by increasing both `NS_number` and the `t_age_max` the simulation would take more time to run.

In [ ]:
cfg["NS_number"] = 100
cfg["t_age_max"] = 3.e7
cfg["B_initial_log10_mean"] = 13.1
cfg["B_initial_log10_sigma"] = 0.45
cfg["P_initial_log10_mean"] = -1.0
cfg["P_initial_log10_sigma"] = 0.38
cfg["a_late"] = -1.80

Alternatively you can change the parameters in the `parameter_override.json` file in the `tutorials/tutorial_notebooks` folder and pass it to the simulator.

In [ ]:
override_dir = "parameter_override.json"

Specify the output directory where the simulation results will be saved.

In [ ]:
output_dir = "output/sim_full"

Run the simulation.

In [ ]:
simulation_args = argparse.Namespace(
    save_dir = output_dir,
    parameter_override = None,
)
simulate_population(simulation_args)

## Read the simulation results

In [ ]:
data_full = pd.read_pickle(
    pathlib.Path().joinpath(output_dir, "final_population.pkl.gz"),
    compression="gzip",
)
data_full.columns

In [ ]:
data_PMPS = pd.read_pickle(
    pathlib.Path().joinpath(output_dir, "survey_PMPS_results.pkl.gz"),
    compression="gzip",
)
data_PMPS.columns

In [ ]:
data_SMPS = pd.read_pickle(
    pathlib.Path().joinpath(output_dir, "survey_SMPS_results.pkl.gz"),
    compression="gzip",
)
data_SMPS.columns

In [ ]:
data_HTRU_low_mid = pd.read_pickle(
    pathlib.Path().joinpath(output_dir, "survey_HTRU_low_mid_results.pkl.gz"),
    compression="gzip",
)
data_HTRU_low_mid.columns

In [ ]:
data_HTRU_high = pd.read_pickle(
    pathlib.Path().joinpath(output_dir, "survey_HTRU_high_results.pkl.gz"),
    compression="gzip",
)
data_HTRU_high.columns

In [ ]:
x = data_full["x"]["[kpc]"].to_numpy()
y = data_full["y"]["[kpc]"].to_numpy()
z = data_full["z"]["[kpc]"].to_numpy()
RA = data_full["RA"]["[deg]"].to_numpy()
DEC = data_full["DEC"]["[deg]"].to_numpy()
pm_RA = data_full["pm_RA"]["[mas yr^-1]"].to_numpy()
pm_DEC = data_full["pm_DEC"]["[mas yr^-1]"].to_numpy()
v_r = data_full["v_r"]["[km s^-1]"].to_numpy()
v_phi = data_full["v_phi"]["[km s^-1]"].to_numpy()
v_z = data_full["v_z"]["[km s^-1]"].to_numpy()
dist = data_full["d"]["[kpc]"].to_numpy()
B = data_full["B"]["[G]"].to_numpy()
chi = data_full["chi"]["[rad]"].to_numpy()
P = data_full["P"]["[s]"].to_numpy()
P_dot = data_full["P_dot"]["[s s^-1]"].to_numpy()
L_radio_bol = data_full["L_radio_bol"]["[erg s^-1]"].to_numpy()
w_int = data_full["w_int"]["[s]"].to_numpy()
intercepted_radio = data_full["intercepted_radio"][" "].to_numpy(dtype=bool)
age = data_full["age"]["[yr]"].to_numpy()
l, b, _, _, _, _ = cc.galactocentric_to_galactic(x, y, z, np.zeros(len(x)), np.zeros(len(x)), np.zeros(len(x)))

In [ ]:
idx_PMPS = data_PMPS["NS_idx"][" "].to_numpy(dtype=int)
S_radio_PMPS = data_PMPS["S_radio_obs_mean"]["[Jy]"].to_numpy()
w_PMPS = data_PMPS["w_eff"]["[s]"].to_numpy()

idx_SMPS = data_SMPS["NS_idx"][" "].to_numpy(dtype=int)
S_radio_SMPS = data_SMPS["S_radio_obs_mean"]["[Jy]"].to_numpy()
w_SMPS = data_SMPS["w_eff"]["[s]"].to_numpy()

idx_HTRU_low_mid = data_HTRU_low_mid["NS_idx"][" "].to_numpy(dtype=int)
S_radio_HTRU_low_mid = data_HTRU_low_mid["S_radio_obs_mean"]["[Jy]"].to_numpy()
w_HTRU_low_mid = data_HTRU_low_mid["w_eff"]["[s]"].to_numpy()

idx_HTRU_high = data_HTRU_high["NS_idx"][" "].to_numpy(dtype=int)
S_radio_HTRU_high = data_HTRU_high["S_radio_obs_mean"]["[Jy]"].to_numpy()
w_HTRU_high = data_HTRU_high["w_eff"]["[s]"].to_numpy()

In [ ]:
set_PMPS = set(idx_PMPS.tolist())
set_SMPS = set(idx_SMPS.tolist())
set_HTRU_low_mid = set(idx_HTRU_low_mid.tolist())
set_HTRU_high = set(idx_HTRU_high.tolist())

idx_radio_detected_noduplicates = list(set_PMPS | set_SMPS | set_HTRU_low_mid | set_HTRU_high)

In [ ]:
number_intercepted = len(intercepted_radio[intercepted_radio == True])
number_detected_PMPS = len(idx_PMPS)
number_detected_SMPS = len(idx_SMPS)
number_detected_HTRU_low_mid = len(idx_HTRU_low_mid)
number_detected_HTRU_high = len(idx_HTRU_high)
number_detected_total = len(idx_radio_detected_noduplicates)

fraction_intercepted = len(intercepted_radio[intercepted_radio == True]) / len(
    intercepted_radio
)
fraction_detected_PMPS = len(idx_PMPS) / len(intercepted_radio)
fraction_detected_SMPS = len(idx_SMPS) / len(intercepted_radio)
fraction_detected_HTRU_low_mid = len(idx_HTRU_low_mid) / len(intercepted_radio)
fraction_detected_HTRU_high = len(idx_HTRU_high) / len(intercepted_radio)

In [ ]:
print(
    f"Total number of detected pulsars: {number_detected_total}"
)
print(
    f"Fraction of pulsars pointing at us: {fraction_intercepted}, ({number_intercepted}/{len(intercepted_radio)})"
)
print(
    f"Fraction of pulsars detected by PMPS: {fraction_detected_PMPS}, ({number_detected_PMPS}/{len(intercepted_radio)})"
)
print(
    f"Fraction of pulsars detected by SMPS: {fraction_detected_SMPS}, ({number_detected_SMPS}/{len(intercepted_radio)})"
)
print(
    f"Fraction of pulsars detected by HTRU low and mid latitude: {fraction_detected_HTRU_low_mid}, ({number_detected_HTRU_low_mid}/{len(intercepted_radio)})"
)
print(
    f"Fraction of pulsars detected by HTRU high latitude: {fraction_detected_HTRU_high}, ({number_detected_HTRU_high}/{len(intercepted_radio)})"
)

## Plot the simulation results

In [ ]:
fig, ax = plt.subplots(figsize=(15, 8))

ax.plot(
    P,
    P_dot,
    linestyle="None",
    marker="o",
    color="lightgray",
    markersize=2,
    alpha=1,
    rasterized=True,
    label="Simulation all",
)
ax.plot(
    P[intercepted_radio],
    P_dot[intercepted_radio],
    linestyle="None",
    marker="o",
    color="black",
    markersize=2,
    alpha=0.3,
    rasterized=True,
    label="Intercepting our LOS",
)
ax.plot(
    P[idx_PMPS],
    P_dot[idx_PMPS],
    linestyle="None",
    marker="o",
    color="tab:red",
    markersize=5,
    alpha=1.0,
    rasterized=True,
    label="Detected by PMPS",
)
ax.plot(
    P[idx_SMPS],
    P_dot[idx_SMPS],
    linestyle="None",
    marker="o",
    color="tab:blue",
    markersize=5,
    alpha=1.0,
    rasterized=True,
    label="Detected by SMPS",
)
ax.plot(
    P[idx_HTRU_low_mid],
    P_dot[idx_HTRU_low_mid],
    linestyle="None",
    marker="o",
    fillstyle="none",
    color="tab:purple",
    markersize=7,
    alpha=1.0,
    rasterized=True,
    label="Detected by HTRU low and mid",
)

ax.set_xscale("log")
ax.set_yscale("log")
# ax.set_ylim(1.e-22, 1.e-20)

plt.xlabel(r"$P$ [s]")
plt.ylabel(r"$\dot{P}$ [s/s]")
plt.legend(
    bbox_to_anchor=(1, 1), 
    frameon=False, 
    loc=0, 
    fontsize=20, 
    markerscale=5, 
    handler_map={PathCollection : HandlerPathCollection(update_func= update), plt.Line2D : HandlerLine2D(update_func = update)}
)

plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(15, 8))

ax.plot(
    l,
    b,
    linestyle="None",
    marker="o",
    color="lightgray",
    markersize=2,
    alpha=1,
    rasterized=True,
    label=r"Simulation all",
)
ax.plot(
    l[intercepted_radio],
    b[intercepted_radio],
    linestyle="None",
    marker="o",
    color="black",
    markersize=2,
    alpha=0.3,
    rasterized=True,
    label=r"Intercepting our LOS",
)
ax.plot(
    l[idx_PMPS],
    b[idx_PMPS],
    linestyle="None",
    marker="o",
    color="tab:red",
    markersize=5,
    alpha=1,
    rasterized=True,
    label=r"Observed PMPS",
)
ax.plot(
    l[idx_SMPS],
    b[idx_SMPS],
    linestyle="None",
    marker="o",
    color="tab:blue",
    markersize=5,
    alpha=1,
    rasterized=True,
    label=r"Observed SMPS",
)
ax.plot(
    l[idx_HTRU_low_mid],
    b[idx_HTRU_low_mid],
    linestyle="None",
    marker="o",
    fillstyle="none",
    color="tab:purple",
    markersize=7,
    alpha=1,
    rasterized=True,
    label=r"Observed HTRU",
)

ax.plot(0.0, 0.0, marker="*", color="tab:orange", markersize=20)
ax.set_xlim(-180.0, 180.0)
ax.set_ylim(-90.0, 90.0)
ax.set_xlabel("l [deg]")
ax.set_ylabel("b [deg]")
plt.legend(bbox_to_anchor=(1, 1), frameon=False, loc=0, fontsize=20)

plt.show()